# 🔷 Chapter 06: Discrete Global Grid Systems (DGGS) & 250m Hexagonal Synthesis
## Overcoming the Modifiable Areal Unit Problem (MAUP) with Hexagonal Bins

### 🎯 The Mission: Why We Are Doing This
Administrative boundaries in Accra vary wildly in size, creating aggregation bias. We generate a uniform **250-meter hexagonal grid (`ST_HexagonGrid`)** to standardize spatial features into isotropic, scale-invariant spatial units.

---

### 🔑 Spatial DBA Keywords Decoded

| Keyword / Concept | Plain-English Explanation | Production Advantage |
| :--- | :--- | :--- |
| **`ST_HexagonGrid(size, bounds)`** | Generates regular hexagons covering the input polygon. | Built into PostGIS kernel (fast, no external dependencies). |
| **MAUP Bias** | Statistical distortion when aggregating data into irregular polygons. | Eliminated by uniform hexagonal lattices. |
| **Isotropic Adjacency** | All 6 neighbor cells have identical centroid distances. | Ideal for spatial weight matrices and GeoAI. |

In [ ]:
from config_db import execute_sql, query_df, query_gdf, preview_map

sql_hex = """
DROP TABLE IF EXISTS accra_study_boundary CASCADE;
CREATE TABLE accra_study_boundary AS
SELECT ST_Buffer(ST_SetSRID(ST_MakePoint(810000, 615000), 32630), 4000.0) AS geom;

DROP TABLE IF EXISTS accra_hex_grid_250m CASCADE;
CREATE TABLE accra_hex_grid_250m AS
SELECT ROW_NUMBER() OVER () AS hex_id, hex.geom::GEOMETRY(Polygon, 32630) AS geom
FROM accra_study_boundary b,
     ST_HexagonGrid(250.0, b.geom) AS hex
WHERE ST_Intersects(hex.geom, b.geom);

CREATE INDEX idx_hex250_geom ON accra_hex_grid_250m USING GIST(geom);

DROP TABLE IF EXISTS accra_hex_density_profile CASCADE;
CREATE TABLE accra_hex_density_profile AS
SELECT 
    h.hex_id, h.geom,
    COUNT(DISTINCT b.bld_id) AS building_count,
    COALESCE(SUM(ST_Area(b.geom)), 0) AS total_built_area_m2,
    ROUND((COALESCE(SUM(ST_Area(b.geom)), 0) / ST_Area(h.geom) * 100.0)::NUMERIC, 2) AS building_coverage_ratio_pct,
    COUNT(DISTINCT l.landmark_id) AS poi_count,
    COALESCE(ROUND((SUM(ST_Length(ST_Intersection(s.geom, h.geom))) / 1000.0)::NUMERIC, 2), 0) AS road_km
FROM accra_hex_grid_250m h
LEFT JOIN riparian_zone_buildings b ON ST_Intersects(b.geom, h.geom)
LEFT JOIN accra_landmarks l ON ST_Intersects(l.geom_utm30n, h.geom)
LEFT JOIN accra_streets s ON ST_Intersects(s.geom, h.geom)
GROUP BY h.hex_id, h.geom;

CREATE INDEX idx_hex_density_geom ON accra_hex_density_profile USING GIST(geom);
"""

execute_sql(sql_hex)
print("✅ Hexagonal 250m urban profile synthesized.")

In [ ]:
# Summary Distribution
df_hex = query_df("""
SELECT hex_id, building_count, building_coverage_ratio_pct, road_km
FROM accra_hex_density_profile
ORDER BY building_coverage_ratio_pct DESC
LIMIT 10;
""")

df_hex

In [ ]:
# Interactive Map Preview of Hexagonal Densities
gdf_hex = query_gdf("""
SELECT hex_id, building_coverage_ratio_pct, road_km, geom FROM accra_hex_density_profile;
""")

preview_map(gdf_hex, tooltip_cols=["hex_id", "building_coverage_ratio_pct", "road_km"])